# 07 - Preprocessing Pipeline & Stratified Splitting

## 1. Zero Data Leakage Guarantee
To prevent data leakage:
- Stratified train-test split (80% train, 20% test, seed=42) is executed **before** any transformations or scaling.
- The test partition is strictly isolated and reserved exclusively for final evaluation.
- All transformations are fit **only** on the training partition.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import RAW_DATA_PATH
from src.data.data_loader import load_and_split_data, save_processed_data
from src.features.preprocessing import build_preprocessing_pipeline

X_train, X_test, y_train, y_test = load_and_split_data(RAW_DATA_PATH)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

2026-10-26 17:56:18 [INFO] data_loader (data_loader.py:42): Loading raw dataset from D:\Personal lone\data\raw\Bank_Personal_Loan_Modelling.csv


2026-10-26 17:56:18 [INFO] data_loader (data_loader.py:44): Loaded raw dataset with shape (5000, 14)


2026-10-26 17:56:18 [INFO] data_validation (validation.py:88): Raw data validation passed successfully.


2026-10-26 17:56:18 [INFO] data_loader (data_loader.py:78): Performing stratified train-test split (test_size=0.20, seed=42)


2026-10-26 17:56:18 [INFO] data_loader (data_loader.py:92): Train shape: (4000, 13), Test shape: (1000, 13)


2026-10-26 17:56:18 [INFO] data_loader (data_loader.py:93): Train target balance: {0: np.int64(3616), 1: np.int64(384)} (9.6% positive)


2026-10-26 17:56:18 [INFO] data_loader (data_loader.py:98): Test target balance: {0: np.int64(904), 1: np.int64(96)} (9.6% positive)


X_train shape: (4000, 13)
X_test shape: (1000, 13)


## 2. Target Stratification Balance Verification

In [2]:
train_prop = y_train.mean() * 100
test_prop = y_test.mean() * 100

print(f"Train positive rate: {train_prop:.2f}% ({y_train.sum()} / {len(y_train)})")
print(f"Test positive rate:  {test_prop:.2f}% ({y_test.sum()} / {len(y_test)})")
assert abs(train_prop - test_prop) < 0.1, "Stratification discrepancy detected!"

Train positive rate: 9.60% (384 / 4000)
Test positive rate:  9.60% (96 / 1000)


## 3. Fit Preprocessing Pipeline on Training Set Only

In [3]:
preprocessor = build_preprocessing_pipeline(use_scaler=False)
X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)

print("Transformed train features shape:", X_train_trans.shape)
print("Transformed test features shape: ", X_test_trans.shape)
display(X_train_trans.head(3))

Transformed train features shape: (4000, 18)
Transformed test features shape:  (1000, 18)


,Age,Experience,Income,Family,CCAvg,Education,Mortgage,Securities Account,CD Account,Online,CreditCard,Income_Per_Family,CCAvg_to_Income_Ratio,Mortgage_to_Income_Ratio,Has_Mortgage,Total_Accounts,Experience_to_Age_Ratio,High_Networth_Indicator
408,60,36,89,2,2.8,1,0,0,0,1,0,44.499978,0.377528,0.000000,0,1,0.600000,0
3737,44,19,30,1,0.5,3,0,0,0,1,0,29.999970,0.200000,0.000000,0,1,0.431818,0
3889,26,0,19,1,0.1,2,121,0,0,1,0,18.999981,0.063158,6.368421,1,1,0.000000,0


## 4. Persist Processed Partitions

In [4]:
save_processed_data(X_train, X_test, y_train, y_test)
print("Processed partitions successfully saved to data/processed/")

2026-10-26 17:56:18 [INFO] data_loader (data_loader.py:127): Persisted processed train data to D:\Personal lone\data\processed\train.csv


2026-10-26 17:56:18 [INFO] data_loader (data_loader.py:128): Persisted processed test data to D:\Personal lone\data\processed\test.csv


Processed partitions successfully saved to data/processed/
